# B3: POS Tagging using a Hidden Markov Model (HMM)

Model POS tagging as an HMM (states = tags, observations = words), estimate transition and emission probabilities, implement the Viterbi algorithm, evaluate tagging accuracy on test sentences, and analyse errors and limitations.

**Dataset:** Penn Treebank sample (3,914 tagged sentences) and the news section of the Brown corpus, both in NLTK with the universal tag set

In [2]:
# nltk : tagged corpora and a ready-made HMM tagger to check ours against
!pip install -q nltk pandas

In [3]:
import math
from collections import Counter, defaultdict
import pandas as pd
import nltk
for pkg in ["treebank", "brown", "universal_tagset"]:
    nltk.download(pkg, quiet=True)
from nltk.corpus import treebank, brown

## Load the data

In [4]:
sents = treebank.tagged_sents(tagset="universal")
train, test = list(sents[:3000]), list(sents[3000:])
print(len(train), "train,", len(test), "test sentences")
print(train[0])

3000 train, 914 test sentences
[('Pierre', 'NOUN'), ('Vinken', 'NOUN'), (',', '.'), ('61', 'NUM'), ('years', 'NOUN'), ('old', 'ADJ'), (',', '.'), ('will', 'VERB'), ('join', 'VERB'), ('the', 'DET'), ('board', 'NOUN'), ('as', 'ADP'), ('a', 'DET'), ('nonexecutive', 'ADJ'), ('director', 'NOUN'), ('Nov.', 'NOUN'), ('29', 'NUM'), ('.', '.')]


## (a) The HMM: states and observations

- **States (hidden):** the tags.
- **Observations:** the words.

In [5]:
tags = sorted({t for s in train for _, t in s})
print(len(tags), "states:", tags)
print(len({w.lower() for s in train for w, _ in s}), "different words (observations)")

12 states: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
9914 different words (observations)


## (b) Transition and emission probabilities

Transition P(tag | previous tag) and emission P(word | tag), counted from the training sentences. `<s>` is the start of a sentence.

**Unknown words:** words seen only once in training are replaced by `<unk>`, and so are new words at test time. The model then learns which tags unknown words usually have. Numbers become `<num>`.

In [6]:
word_freq = Counter(w.lower() for s in train for w, _ in s)
def norm(w):
    w = w.lower()
    if w[0].isdigit(): return "<num>"                  # 1990, 3.5, 14.99
    return w if word_freq[w] > 1 else "<unk>"

trans_counts = Counter()
emit_counts = Counter()
tag_counts = Counter()
for s in train:
    prev = "<s>"
    for word, tag in s:
        trans_counts[(prev, tag)] += 1
        emit_counts[(tag, norm(word))] += 1
        tag_counts[tag] += 1
        prev = tag

trans = pd.DataFrame(0.0, index=["<s>"] + tags, columns=tags)
for (p, t), c in trans_counts.items():
    trans.loc[p, t] = c
trans = trans.div(trans.sum(axis=1), axis=0)          # each row adds up to 1
trans.round(2)

,.,ADJ,ADP,ADV,CONJ,DET,NOUN,NUM,PRON,PRT,VERB,X
<s>,0.09,0.04,0.13,0.06,0.06,0.23,0.26,0.01,0.08,0.00,0.01,0.03
.,0.11,0.04,0.07,0.05,0.06,0.14,0.19,0.10,0.07,0.00,0.13,0.03
ADJ,0.06,0.07,0.07,0.01,0.02,0.00,0.70,0.02,0.00,0.01,0.01,0.02
ADP,0.03,0.11,0.02,0.01,0.00,0.32,0.32,0.06,0.08,0.00,0.01,0.04
ADV,0.14,0.13,0.12,0.08,0.01,0.07,0.03,0.02,0.02,0.01,0.34,0.02
CONJ,0.03,0.13,0.05,0.06,0.00,0.12,0.35,0.03,0.07,0.01,0.16,0.01
DET,0.02,0.21,0.01,0.01,0.00,0.01,0.64,0.02,0.00,0.00,0.04,0.04
NOUN,0.25,0.01,0.18,0.02,0.04,0.01,0.26,0.01,0.01,0.04,0.14,0.03
NUM,0.12,0.04,0.04,0.00,0.02,0.00,0.36,0.16,0.00,0.02,0.02,0.21
PRON,0.04,0.07,0.02,0.04,0.01,0.01,0.20,0.01,0.01,0.01,0.48,0.09


In [7]:
def emission(tag, word):
    return emit_counts[(tag, word)] / tag_counts[tag]

for w in ["race", "book", "the", "can", "<unk>"]:
    print(f"{w:<5}", {t: round(emission(t, w), 4) for t in tags if emission(t, w) > 0})

race  {'NOUN': 0.0001}
book  {'NOUN': 0.0002, 'VERB': 0.0001}
the   {'ADJ': 0.001, 'DET': 0.5452, 'NOUN': 0.0002, 'NUM': 0.0004}
can   {'NOUN': 0.0, 'VERB': 0.0081}
<unk> {'ADJ': 0.1631, 'ADP': 0.0007, 'ADV': 0.047, 'CONJ': 0.0006, 'DET': 0.0003, 'NOUN': 0.1086, 'NUM': 0.0043, 'PRON': 0.0004, 'VERB': 0.1025, 'X': 0.0523}


## (c) The Viterbi algorithm

For each word and each tag, keep the probability of the best tag sequence ending in that tag, and a back-pointer. Logs avoid multiplying many small numbers. Add-k smoothing (k = 0.01) gives unseen events a small probability.

In [8]:
V = len({w for (_, w) in emit_counts}) + 1
def log_trans(p, t, k=0.01):
    return math.log((trans_counts[(p, t)] + k) / (sum(trans_counts[(p, x)] for x in tags) + k * len(tags)))
def log_emit(t, w, k=0.01):
    return math.log((emit_counts[(t, w)] + k) / (tag_counts[t] + k * V))

def viterbi(words):
    words = [norm(w) for w in words]
    best = [{t: log_trans("<s>", t) + log_emit(t, words[0]) for t in tags}]   # first word
    back = [{}]
    for i in range(1, len(words)):
        best.append({}); back.append({})
        for t in tags:
            prev = max(tags, key=lambda p: best[i-1][p] + log_trans(p, t))       # best previous tag
            best[i][t] = best[i-1][prev] + log_trans(prev, t) + log_emit(t, words[i])
            back[i][t] = prev
    last = max(tags, key=lambda t: best[-1][t])
    path = [last]
    for i in range(len(words) - 1, 0, -1):                                    # follow the back-pointers
        path.insert(0, back[i][path[0]])
    return path

sentence = "They can book a race".split()
list(zip(sentence, viterbi(sentence)))

[('They', 'PRON'),
 ('can', 'VERB'),
 ('book', 'VERB'),
 ('a', 'DET'),
 ('race', 'NOUN')]

In [9]:
# NLTK's HMM tagger (also uses Viterbi), to check ours
from nltk.tag import hmm
from nltk.probability import LidstoneProbDist
nltk_hmm = hmm.HiddenMarkovModelTrainer().train_supervised(train, estimator=lambda fd, bins: LidstoneProbDist(fd, 0.01, bins))
nltk_hmm.tag(sentence)

[('They', 'PRON'),
 ('can', 'VERB'),
 ('book', 'VERB'),
 ('a', 'DET'),
 ('race', 'NOUN')]

## (d) Accuracy on the test sentences

In [10]:
def evaluate(test_sents, tagger):
    gold, pred, words = [], [], []
    for s in test_sents:
        ws = [w for w, _ in s]
        gold += [t for _, t in s]; pred += tagger(ws); words += ws
    return gold, pred, words

gold, pred, words = evaluate(test, viterbi)
acc = sum(g == p for g, p in zip(gold, pred)) / len(gold)
print(f"our HMM on Penn Treebank test: {acc:.1%}")
print(f"NLTK HMM:                      {nltk_hmm.accuracy(test):.1%}")

our HMM on Penn Treebank test: 93.0%


NLTK HMM:                      90.8%


In [11]:
# baseline: give every word its most frequent tag in training (NOUN for unknown words)
most_freq = {}
for (t, w), c in emit_counts.items():
    if c > emit_counts[(most_freq.get(w), w)]:
        most_freq[w] = t
base = [most_freq.get(norm(w), "NOUN") for w in words]
print(f"most-frequent-tag baseline: {sum(g == b for g, b in zip(gold, base)) / len(gold):.1%}")

most-frequent-tag baseline: 91.8%


## (e) Error analysis

In [12]:
errors = Counter((g, p) for g, p in zip(gold, pred) if g != p)
pd.DataFrame(errors.most_common(8), columns=["(gold, predicted)", "count"])

,"(gold, predicted)",count
0,"(NOUN, ADJ)",423
1,"(VERB, NOUN)",202
2,"(NOUN, VERB)",164
3,"(ADJ, NOUN)",160
4,"(VERB, ADJ)",96
5,"(NUM, X)",77
6,"(X, NUM)",61
7,"(ADJ, ADV)",46


In [13]:
unk = [norm(w) == "<unk>" for w in words]
acc_known = sum(g == p for g, p, u in zip(gold, pred, unk) if not u) / unk.count(False)
acc_unk = sum(g == p for g, p, u in zip(gold, pred, unk) if u) / unk.count(True)
print(f"known words: {acc_known:.1%}   unknown words: {acc_unk:.1%}   ({unk.count(True)} unknown tokens)")
print("wrongly tagged unknown words:", [(w, g, p) for w, g, p, u in zip(words, gold, pred, unk) if u and g != p][:8])

known words: 95.7%   unknown words: 69.4%   (2379 unknown tokens)
wrongly tagged unknown words: [('Nikkei', 'NOUN', 'ADJ'), ('selected', 'VERB', 'NOUN'), ('Nikkei', 'NOUN', 'ADJ'), ('outnumbered', 'VERB', 'ADJ'), ('advancing', 'VERB', 'NOUN'), ('switched', 'VERB', 'NOUN'), ('sparked', 'VERB', 'NOUN'), ('incentive-backed', 'ADJ', 'NOUN')]


## The Brown corpus (news section), with NLTK's HMM tagger (no special handling of unknown words)

In [14]:
b = list(brown.tagged_sents(categories="news", tagset="universal"))
cut = int(0.9 * len(b))
nltk_brown = hmm.HiddenMarkovModelTrainer().train_supervised(b[:cut], estimator=lambda fd, bins: LidstoneProbDist(fd, 0.01, bins))
print(f"Brown news: {len(b[:cut])} train, {len(b[cut:])} test sentences, HMM accuracy {nltk_brown.accuracy(b[cut:]):.1%}")

Brown news: 4160 train, 463 test sentences, HMM accuracy 88.1%


## Limitations of the HMM
- Each tag depends only on the previous tag; longer context is ignored.
- Each word depends only on its own tag; spelling clues (capital letter, -ing, -ly) are not used, so **unknown words** are tagged much worse than known ones.
- Most errors are between similar tags (NOUN / ADJ / VERB), where one tag of context is not enough.
- It needs hand-tagged training data, and accuracy drops on text unlike the training data.